# 08. Stress vs calm -- paired test (Table A.19)

Formal statistical test: is cross-frequency ARI different between the
stress (wartime) and calm (peacetime) reference windows?

This is a *post-bootstrap* test. Its inputs are the calm/stress ARIs
already computed by notebook 03 (per asset, per episode), pooled across
the 6 asset-by-episode pairs and tested for paired difference. The
notebook walks through the pooling and runs the three tests
(paired t, Wilcoxon signed-rank, exact sign) directly via scipy.

**Re-run command**: `python run.py stress_vs_calm`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- pool calm/stress ARIs from cached bootstrap CSVs

In [2]:
from src.experiments.exp_07_stress_vs_calm import load_pairs

out_dirs = [(OUT, "2026_US_Iran")]
if (OUT_2022 / "bootstrap_five_day_windows.csv").exists():
    out_dirs.append((OUT_2022, "2022_Russia_Ukraine"))

full = load_pairs(out_dirs)
display(full.round(4))

,episode,asset,calm_ARI,stress_ARI,diff_stress_minus_calm,boot_median,boot_q025,boot_q975,p_calm_vs_boot,p_stress_vs_boot
0,2026_US_Iran,SPY,0.097,0.190,0.093,0.109,0.063,0.265,0.779,0.158
1,2026_US_Iran,USDJPY,0.079,0.052,-0.027,0.081,-0.006,0.456,0.962,0.553
2,2026_US_Iran,CL,0.136,0.070,-0.066,0.164,-0.006,0.500,0.786,0.362
3,2026_US_Iran,GLD,0.147,0.143,-0.005,0.131,0.042,0.300,0.755,0.814
4,2022_Russia_Ukraine,SPY,0.106,0.120,0.014,0.119,0.079,0.264,0.760,0.992
5,2022_Russia_Ukraine,USDJPY,0.094,0.103,0.009,0.110,0.063,0.286,0.755,0.904
6,2022_Russia_Ukraine,GLD,0.050,0.128,0.077,0.114,0.014,0.294,0.244,0.687


## Step 2 -- paired t-test, Wilcoxon, exact sign test

In [3]:
from scipy import stats

calm = full["calm_ARI"].values
stress = full["stress_ARI"].values
diffs = stress - calm

t_stat, p_t = stats.ttest_rel(stress, calm)
try:
    w_stat, p_w = stats.wilcoxon(stress, calm)
except Exception:
    w_stat, p_w = float("nan"), float("nan")
n = len(diffs); n_gt = int((diffs > 0).sum())
p_sign = stats.binomtest(n_gt, n, p=0.5, alternative="two-sided").pvalue

pd.DataFrame([{
    "n_pairs": n,
    "mean_diff_stress_minus_calm": float(diffs.mean()),
    "stress_gt_calm": n_gt,
    "paired_t": float(t_stat), "paired_t_p": float(p_t),
    "wilcoxon": float(w_stat), "wilcoxon_p": float(p_w),
    "sign_p": float(p_sign),
}]).round(4)

,n_pairs,mean_diff_stress_minus_calm,stress_gt_calm,paired_t,paired_t_p,wilcoxon,wilcoxon_p,sign_p
0,7,0.014,4,0.65,0.54,10.0,0.578,1.0


## Cached summary -- `outputs/stress_vs_calm_test_summary.csv`

In [4]:
p = OUT / "stress_vs_calm_test_summary.csv"
display(pd.read_csv(p).round(4) if p.exists() else Markdown(f"`{p}` missing"))

,n_pairs,mean_diff,std_diff,median_diff,stress_gt_calm_count,paired_t_stat,paired_t_p,wilcoxon_stat,wilcoxon_p,sign_test_p,all_boot_p_gt_0_3,min_boot_p
0,7,0.014,0.056,0.009,4,0.65,0.54,10.0,0.578,1.0,False,0.158


## Plain-text report -- `outputs/stress_vs_calm_test.txt`

In [5]:
p = OUT / "stress_vs_calm_test.txt"
print(p.read_text() if p.exists() else f"{p} missing")

Formal test: does wartime differ from peacetime in cross-frequency ARI?

Paired data (7 asset-by-episode pairs):
  ----------------------------------------------------------------------
  episode               asset        calm   stress      diff
  ----------------------------------------------------------------------
  2026_US_Iran          SPY        0.0966   0.1896   +0.0930
  2026_US_Iran          USDJPY     0.0786   0.0521   -0.0266
  2026_US_Iran          CL         0.1363   0.0705   -0.0658
  2026_US_Iran          GLD        0.1475   0.1427   -0.0048
  2022_Russia_Ukraine   SPY        0.1063   0.1202   +0.0139
  2022_Russia_Ukraine   USDJPY     0.0937   0.1026   +0.0089
  2022_Russia_Ukraine   GLD        0.0504   0.1277   +0.0773

n = 7
mean(stress - calm) = +0.0137
std(stress - calm)  = 0.0558
median diff         = +0.0089
stress > calm in 4/7 pairs

Hypothesis tests (H0: stress ARI = calm ARI):
  Paired t-test :  t = +0.650, p = 0.540
  Wilcoxon      :  W = 10.00, p = 0.578
  